In [1]:
# Baseline OLS regressions: Table 2, Tables C.5 and C.6, Section 4.3
import os
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

warnings.filterwarnings("ignore")
os.makedirs("output", exist_ok=True)

In [2]:
# Merge the clean files into one monthly data set, December 2015 to December 2025 (121 months)
files = ["spy_clean.csv", "rsp_clean.csv", "voo_clean.csv", "vix_clean.csv", "dgs10_clean.csv",
         "AIStocks_ASVI8_Monthly.csv", "AIStocks_ASVI4_Monthly.csv",
         "mag7_clean.csv", "mag7b_clean.csv", "mag7c_clean.csv"]
df = pd.read_csv(files[0], parse_dates=["date"])
for f in files[1:]:
    df = df.merge(pd.read_csv(f, parse_dates=["date"]), on="date", how="inner")
df = df.sort_values("date").reset_index(drop=True)

# 2021-2025 subsample. December 2020 only serves as the first lag, so the VARs run on Jan 2021 to Dec 2025.
sub = df[df["date"] >= "2020-12-31"].reset_index(drop=True)
print(len(df), "months,", df["date"].min().date(), "to", df["date"].max().date())

121 months, 2015-12-31 to 2025-12-31


In [3]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def ols(data, y, x):
    # Constant, regressor of interest and the two macro controls, HC3 robust standard errors
    X = sm.add_constant(data[[x, "dlog_vix", "d_dgs10"]])
    return sm.OLS(data[y], X).fit(cov_type="HC3")

In [4]:
# One-month lags. December 2015 only provides the lag, so the regressions run on Jan 2016 to Dec 2025 (N = 120).
d = df.copy()
for c in ["ASVI8", "flow_spy", "flow_rsp", "flow_voo", "mag7_ret"]:
    d[c + "_lag"] = d[c].shift(1)
d = d.dropna().reset_index(drop=True)

# Sample windows for Tables C.5 and C.6
windows = {"2016-2025": "2016-01-31", "2020-2025": "2020-01-31", "2021-2025": "2021-01-31",
           "2022-2025": "2022-01-31", "2023-2025": "2023-01-31"}
window = {w: d[d["date"] >= start] for w, start in windows.items()}
print({w: len(data) for w, data in window.items()})

{'2016-2025': 120, '2020-2025': 72, '2021-2025': 60, '2022-2025': 48, '2023-2025': 36}


In [5]:
# Table 2: full sample
models = {
    "H1": ols(d, "flow_spy", "ASVI8_lag"),            # attention -> SPY flows
    "H1b": ols(d, "mag7_ret", "flow_spy_lag"),        # SPY flows -> Mag7 returns
    "H2": ols(d, "mag7_ret", "ASVI8_lag"),            # attention -> Mag7 returns
    "Feedback": ols(d, "ASVI8", "mag7_ret_lag"),      # Mag7 returns -> attention
}
for name, m in models.items():
    print(name)
    print(m.summary().tables[1], "\n")

rows = {"ASVI8_lag": "ASVI8 lag(1)", "flow_spy_lag": "Flow SPY lag(1)", "mag7_ret_lag": "Mag7 returns lag(1)",
        "dlog_vix": "dlogVIX", "d_dgs10": "dDGS10"}
table2 = {}
for name, m in models.items():
    col = []
    for var in rows:
        if var in m.params:
            col += [f"{m.params[var]:.3f}{stars(m.pvalues[var])}", f"({m.bse[var]:.3f})"]
        else:
            col += ["", ""]
    col += [int(m.nobs), f"{m.rsquared:.3f}", f"{durbin_watson(m.resid):.3f}", f"{m.fvalue:.4g}"]
    table2[name] = col
index = [x for v in rows.values() for x in (v, "")] + ["N", "R2", "Durbin-Watson", "F-statistic"]
table2 = pd.DataFrame(table2, index=index)
table2.to_csv("output/Table_2_OLS_full_sample.csv")
table2

H1
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0017      0.003      0.649      0.517      -0.003       0.007
ASVI8_lag     -0.0053      0.004     -1.325      0.185      -0.013       0.003
dlog_vix      -0.0331      0.012     -2.858      0.004      -0.056      -0.010
d_dgs10        0.0033      0.011      0.294      0.769      -0.019       0.026

H1b
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0234      0.006      3.901      0.000       0.012       0.035
flow_spy_lag     0.2708      0.218      1.244      0.214      -0.156       0.698
dlog_vix        -0.1572      0.027     -5.897      0.000      -0.209      -0.105
d_dgs10         -0.0463      0.028     -1.673      0.094      -0.101       0.008

H2
                 coef    std

,H1,H1b,H2,Feedback
ASVI8 lag(1),-0.005,,0.023**,
,(0.004),,(0.011),
Flow SPY lag(1),,0.271,,
,,(0.218),,
Mag7 returns lag(1),,,,2.311***
,,,,(0.886)
dlogVIX,-0.033***,-0.157***,-0.152***,-0.167
,(0.012),(0.027),(0.026),(0.210)
dDGS10,0.003,-0.046*,-0.041,-0.180
,(0.011),(0.028),(0.027),(0.233)


In [6]:
# Tables C.5 and C.6: all funds and sample windows
def cell(m, var, with_stars=False):
    return f"{m.params[var]:.4f}{stars(m.pvalues[var]) if with_stars else ''} ({m.bse[var]:.4f})"

funds = ["spy", "rsp", "voo"]
h1 = {(w, f): ols(window[w], f"flow_{f}", "ASVI8_lag") for w in windows for f in funds}
h1b = {(w, f): ols(window[w], "mag7_ret", f"flow_{f}_lag") for w in windows for f in funds}
h2 = {w: ols(window[w], "mag7_ret", "ASVI8_lag") for w in windows}
fb = {w: ols(window[w], "ASVI8", "mag7_ret_lag") for w in windows}

rowsC5 = []
for w in windows:
    rowsC5.append({"Panel": "A: Flow on ASVI8(t-1)", "Sample window": w, "N": len(window[w]),
                   **{f.upper(): cell(h1[(w, f)], "ASVI8_lag") for f in funds}})
for w in windows:
    rowsC5.append({"Panel": "B: Mag7 returns on Flow(t-1)", "Sample window": w, "N": len(window[w]),
                   **{f.upper(): cell(h1b[(w, f)], f"flow_{f}_lag") for f in funds}})
tableC5 = pd.DataFrame(rowsC5)
tableC5.to_csv("output/Table_C5_OLS_flows_by_window.csv", index=False)

tableC6 = pd.DataFrame([{
    "Sample window": w, "N": len(window[w]),
    "H2: beta on ASVI8(t-1)": cell(h2[w], "ASVI8_lag", True), "H2 R2": f"{h2[w].rsquared:.3f}",
    "H2 DW": f"{durbin_watson(h2[w].resid):.2f}",
    "Feedback: beta on Mag7(t-1)": cell(fb[w], "mag7_ret_lag", True), "Feedback R2": f"{fb[w].rsquared:.3f}",
    "Feedback DW": f"{durbin_watson(fb[w].resid):.2f}",
} for w in windows])
tableC6.to_csv("output/Table_C6_OLS_returns_and_feedback.csv", index=False)

dw_a = [durbin_watson(m.resid) for m in h1.values()]
dw_b = [durbin_watson(m.resid) for m in h1b.values()]
print(f"Durbin-Watson: Panel A {min(dw_a):.2f} to {max(dw_a):.2f}, Panel B {min(dw_b):.2f} to {max(dw_b):.2f}")
display(tableC5)
tableC6

Durbin-Watson: Panel A 1.37 to 2.72, Panel B 1.59 to 1.96


,Panel,Sample window,N,SPY,RSP,VOO
0,A: Flow on ASVI8(t-1),2016-2025,120,-0.0053 (0.0040),-0.0000 (0.0078),0.0008 (0.0026)
1,A: Flow on ASVI8(t-1),2020-2025,72,-0.0048 (0.0041),-0.0014 (0.0094),0.0013 (0.0032)
2,A: Flow on ASVI8(t-1),2021-2025,60,-0.0032 (0.0041),0.0022 (0.0102),0.0009 (0.0029)
3,A: Flow on ASVI8(t-1),2022-2025,48,-0.0028 (0.0048),-0.0016 (0.0117),-0.0004 (0.0029)
4,A: Flow on ASVI8(t-1),2023-2025,36,-0.0049 (0.0057),-0.0021 (0.0140),-0.0032 (0.0025)
5,B: Mag7 returns on Flow(t-1),2016-2025,120,0.2708 (0.2178),-0.0163 (0.1768),-0.0927 (0.3810)
6,B: Mag7 returns on Flow(t-1),2020-2025,72,0.1766 (0.3164),-0.1139 (0.2088),0.0997 (0.5556)
7,B: Mag7 returns on Flow(t-1),2021-2025,60,-0.1778 (0.3196),0.1045 (0.2080),0.3658 (0.7495)
8,B: Mag7 returns on Flow(t-1),2022-2025,48,-0.1803 (0.3423),-0.0050 (0.2812),0.7257 (0.9134)
9,B: Mag7 returns on Flow(t-1),2023-2025,36,-0.4550 (0.4251),-0.0875 (0.2265),-0.8263 (1.0422)


,Sample window,N,H2: beta on ASVI8(t-1),H2 R2,H2 DW,Feedback: beta on Mag7(t-1),Feedback R2,Feedback DW
0,2016-2025,120,0.0231** (0.0106),0.287,2.04,2.3110*** (0.8857),0.092,0.51
1,2020-2025,72,0.0242* (0.0124),0.330,2.09,2.8419** (1.1391),0.142,0.50
2,2021-2025,60,0.0267** (0.0131),0.376,2.04,3.7016** (1.4713),0.174,0.57
3,2022-2025,48,0.0330** (0.0139),0.459,2.08,4.5415*** (1.4466),0.296,0.71
4,2023-2025,36,0.0140 (0.0158),0.265,1.79,4.4123* (2.6604),0.206,0.41


In [7]:
# Numbers in Section 4.3
m = h1[("2016-2025", "spy")]
print(f"H1 SPY: beta {m.params['ASVI8_lag']:.4f}, t {m.tvalues['ASVI8_lag']:.2f}, p {m.pvalues['ASVI8_lag']:.3f}")
for f in funds:
    print(f"H1 {f.upper()}: positive in", sum(h1[(w, f)].params["ASVI8_lag"] > 0 for w in windows), "of 5 windows")
print("H1: significant at 10% in", sum(m.pvalues["ASVI8_lag"] < 0.10 for m in h1.values()), "of 15 regressions")

print()
for w in windows:
    print(f"{w}:  H1b SPY p {h1b[(w, 'spy')].pvalues['flow_spy_lag']:.3f}  |  "
          f"H2 beta {h2[w].params['ASVI8_lag']:.4f}, p {h2[w].pvalues['ASVI8_lag']:.3f}  |  "
          f"Feedback p {fb[w].pvalues['mag7_ret_lag']:.3f}")

H1 SPY: beta -0.0053, t -1.32, p 0.185
H1 SPY: positive in 0 of 5 windows
H1 RSP: positive in 1 of 5 windows
H1 VOO: positive in 3 of 5 windows
H1: significant at 10% in 0 of 15 regressions

2016-2025:  H1b SPY p 0.214  |  H2 beta 0.0231, p 0.029  |  Feedback p 0.009
2020-2025:  H1b SPY p 0.577  |  H2 beta 0.0242, p 0.051  |  Feedback p 0.013
2021-2025:  H1b SPY p 0.578  |  H2 beta 0.0267, p 0.041  |  Feedback p 0.012
2022-2025:  H1b SPY p 0.598  |  H2 beta 0.0330, p 0.018  |  Feedback p 0.002
2023-2025:  H1b SPY p 0.284  |  H2 beta 0.0140, p 0.378  |  Feedback p 0.097
